<a href="https://colab.research.google.com/github/gabrielcabanilla1/EXSU-500-Clinical-NLP-large-language-models-rct/blob/main/notebooks/03c_pubmedbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03c – PubMedBERT (deep learning model)
Fine-tunes PubMedBERT (Gu et al., 2021) to classify each sentence of an RCT abstract.
Two versions: sentence only (USE_CONTEXT = False) and sentence + position + neighbouring
sentences (USE_CONTEXT = True). Input: the files prepared by 02_data_preparation.
Only the training and validation splits are used here; the test set is evaluated once, in 06_final_test.

In [2]:
# ===== CONFIGURATION =====
USE_CONTEXT = False
SUBSAMPLE = None
SEED = 42

DATA_VERSION = 'PubMed_20k_RCT'
PROJECT_DIR = '/content/drive/MyDrive/exsu_project'
DATA_DIR = f'{PROJECT_DIR}/prepared_data'
MODEL_NAME = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
CLASSES = ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']

EPOCHS = 2
LEARNING_RATE = 3e-5
if USE_CONTEXT:
    BATCH_SIZE, MAX_LENGTH, RUN_NAME = 32, 256, 'pubmedbert_context'
else:
    BATCH_SIZE, MAX_LENGTH, RUN_NAME = 64, 128, 'pubmedbert_text'

MODEL_DIR = f'{PROJECT_DIR}/models/{RUN_NAME}'
PREDICTIONS_PATH = f'{PROJECT_DIR}/predictions/dev_{RUN_NAME}.csv'
print('Run:', RUN_NAME, '| batch size:', BATCH_SIZE, '| max length:', MAX_LENGTH)

Run: pubmedbert_text | batch size: 64 | max length: 128


In [3]:
import os
import torch
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(os.path.dirname(PREDICTIONS_PATH), exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, '|', torch.cuda.get_device_name(0) if device == 'cuda' else 'NO GPU')

def load_split(split):
    return pd.read_csv(f'{DATA_DIR}/{DATA_VERSION}_{split}.csv',
                       dtype={'abstract_id': str}, keep_default_na=False)

train, dev = load_split('train'), load_split('dev')
assert len(train) == 180040 and len(dev) == 30212, 'Unexpected size: re-run 02_data_preparation'
assert not set(train.abstract_id) & set(dev.abstract_id), 'Leakage: abstract in train and dev'
print(f'Train: {len(train):,} sentences | Dev: {len(dev):,} sentences')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda | NVIDIA L4
Train: 180,040 sentences | Dev: 30,212 sentences


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenise(df):
    if USE_CONTEXT:
        # Sentence + context as a text pair; the longer segment is trimmed first
        return tokenizer(list(df.text), list(df.context), truncation='longest_first',
                         max_length=MAX_LENGTH, padding='max_length', return_tensors='pt')
    return tokenizer(list(df.text), truncation=True, max_length=MAX_LENGTH,
                     padding='max_length', return_tensors='pt')

train_run = train.sample(SUBSAMPLE, random_state=SEED) if SUBSAMPLE else train
tokens_train, tokens_dev = tokenise(train_run), tokenise(dev)

print('Training sentences:', len(train_run))
print('Example tokens:', tokenizer.convert_ids_to_tokens(tokens_train['input_ids'][0][:15]))

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

Training sentences: 180040
Example tokens: ['[CLS]', 'to', 'investigate', 'the', 'efficacy', 'of', '6', 'weeks', 'of', 'daily', 'low', '-', 'dose', 'oral', 'prednisolone']


In [5]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(SEED)
np.random.seed(SEED)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(CLASSES)).to(device)

train_dataset = TensorDataset(tokens_train['input_ids'], tokens_train['attention_mask'],
                              torch.tensor(train_run.y.values))
dev_dataset = TensorDataset(tokens_dev['input_ids'], tokens_dev['attention_mask'])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))

optimiser = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimiser, int(0.06 * total_steps), total_steps)
scaler = torch.amp.GradScaler(enabled=(device == 'cuda'))

@torch.no_grad()
def predict(dataset):
    """Return the probability of each class for every sentence (n x 5)."""
    model.eval()
    probs = []
    for ids, mask in DataLoader(dataset, batch_size=256):
        with torch.autocast(device, enabled=(device == 'cuda')):
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
        probs.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(probs)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ide

In [6]:
import time, json
from tqdm.auto import tqdm
from sklearn.metrics import f1_score

best_f1, start, log = -1, time.time(), []

for epoch in range(EPOCHS):
    model.train()
    for ids, mask, y_batch in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{EPOCHS}'):
        optimiser.zero_grad()
        with torch.autocast(device, enabled=(device == 'cuda')):
            loss = model(input_ids=ids.to(device), attention_mask=mask.to(device),
                         labels=y_batch.to(device)).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimiser)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimiser)
        scaler.update()
        scheduler.step()

    dev_f1 = f1_score(dev.y, predict(dev_dataset).argmax(1), average='macro')
    minutes = (time.time() - start) / 60
    log.append({'epoch': epoch + 1, 'dev_macro_f1': round(dev_f1, 4), 'minutes': round(minutes, 1)})
    print(f'Epoch {epoch + 1}: dev macro-F1 = {dev_f1:.4f} | {minutes:.1f} min')
    if dev_f1 > best_f1:
        best_f1 = dev_f1
        model.save_pretrained(MODEL_DIR)
        tokenizer.save_pretrained(MODEL_DIR)
        print('  → Best so far, saved to Drive')

run_info = {'run': RUN_NAME, 'model': MODEL_NAME, 'use_context': USE_CONTEXT, 'subsample': SUBSAMPLE,
            'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'max_length': MAX_LENGTH,
            'learning_rate': LEARNING_RATE, 'seed': SEED,
            'gpu': torch.cuda.get_device_name(0) if device == 'cuda' else 'cpu',
            'total_minutes': round((time.time() - start) / 60, 1), 'log': log}
json.dump(run_info, open(f'{MODEL_DIR}/run_info.json', 'w'), indent=2)
print(f"\nTotal training time: {run_info['total_minutes']} minutes")

Epoch 1/2:   0%|          | 0/2814 [00:00<?, ?it/s]

Epoch 1: dev macro-F1 = 0.8342 | 10.2 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Best so far, saved to Drive


Epoch 2/2:   0%|          | 0/2814 [00:00<?, ?it/s]

Epoch 2: dev macro-F1 = 0.8365 | 20.3 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Best so far, saved to Drive

Total training time: 20.4 minutes


In [7]:
from sklearn.metrics import accuracy_score, classification_report

model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
dev_probs = predict(dev_dataset)
dev_pred = dev_probs.argmax(1)

print('Accuracy:', round(accuracy_score(dev.y, dev_pred), 3))
print('Macro-F1:', round(f1_score(dev.y, dev_pred, average='macro'), 3))
print()
print(classification_report(dev.y, dev_pred, target_names=CLASSES, digits=3))

output = dev[['abstract_id', 'sentence_idx', 'label']].copy()
output['prediction'] = [CLASSES[i] for i in dev_pred]
for i, c in enumerate(CLASSES):
    output[f'prob_{c}'] = dev_probs[:, i]
output.to_csv(PREDICTIONS_PATH, index=False)
print('Predictions saved to', PREDICTIONS_PATH)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Accuracy: 0.89
Macro-F1: 0.837

              precision    recall  f1-score   support

  BACKGROUND      0.694     0.836     0.758      3449
   OBJECTIVE      0.787     0.593     0.676      2376
     METHODS      0.943     0.962     0.952      9964
     RESULTS      0.941     0.927     0.934      9841
 CONCLUSIONS      0.878     0.846     0.862      4582

    accuracy                          0.890     30212
   macro avg      0.849     0.833     0.837     30212
weighted avg      0.892     0.890     0.889     30212

Predictions saved to /content/drive/MyDrive/exsu_project/predictions/dev_pubmedbert_text.csv
